# **Activate GPU**

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# **Add Important Libraries**

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout,Rescaling,RandomFlip,RandomRotation,RandomZoom,BatchNormalization,GlobalAveragePooling2D,RandomContrast,RandomBrightness,Activation


from tensorflow.keras.initializers import HeNormal
from tensorflow.keras import layers
from tensorflow.data import AUTOTUNE
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from tensorflow.keras.models import load_model

# **Add Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# **Add Kaggle API**

In [ ]:
from google.colab import files
files.upload()



# **Add Dataset From Kaggle**

In [ ]:
!mkdir -p ~/.kaggle


!mv kaggle.json ~/.kaggle/


!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d rizwan123456789/potato-disease-leaf-datasetpld

!unzip -q potato-disease-leaf-datasetpld.zip -d ./data

!rm potato-disease-leaf-datasetpld.zip

!ls ./data

Dataset URL: https://www.kaggle.com/datasets/rizwan123456789/potato-disease-leaf-datasetpld
License(s): DbCL-1.0
  0% 0.00/37.4M [00:00<?, ?B/s]
100% 37.4M/37.4M [00:00<00:00, 1.59GB/s]
PLD_3_Classes_256


# **Add pretrained model VGG**

In [ ]:
from tensorflow.keras.applications import VGG19
base_model_vgg=VGG19(include_top=False,input_shape=(224,224,3),weights='imagenet')

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


# **Set Data Directory**

In [ ]:
data_dir = "/content/data/PLD_3_Classes_256"

train_dir = os.path.join(data_dir, "Training")
val_dir   = os.path.join(data_dir, "Validation")
test_dir  = os.path.join(data_dir, "Testing")

# **Split The Data into Train and Test Set**

In [ ]:
train_ds=keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224,224),
    batch_size=64,
    shuffle=True,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)
val_ds=keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(224,224),
    batch_size=64,
    shuffle=False,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)
test_ds=keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224,224),
    batch_size=64,
    shuffle=False,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)

Found 3251 files belonging to 3 classes.
Found 416 files belonging to 3 classes.
Found 405 files belonging to 3 classes.


## **Find The Class Name**

In [ ]:
class_names=train_ds.class_names
print(class_names)

['Early_Blight', 'Healthy', 'Late_Blight']


# **Perform Cache and Autotune for Fast Processing**

In [ ]:
train_ds=train_ds.cache().shuffle(2500).prefetch(buffer_size=AUTOTUNE)
test_ds=test_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds=val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# **Perform Data Augmentation to Reduce Overfitting**

In [ ]:
data_augmentation = keras.Sequential(
    [
        RandomFlip("horizontal"),
        RandomRotation(0.1),
        RandomZoom(0.2),
    ]
)

# **Freeze Upper LAyer and Add Custom Dense Layer**

In [ ]:
#Freeze all layers
for layer in base_model_vgg.layers:
  layer.trainable = False


#unfreeze last 100 layers
for layer in base_model_vgg.layers[-50:]:
  layer.trainable=True



inp=layers.Input(shape=(224,224,3))

x=data_augmentation(inp)
x = base_model_vgg(x, training=False)

#add own fully connected layers

x=GlobalAveragePooling2D()(x)
x=Dense(128,activation="relu",kernel_initializer='he_normal')(x)
x=Dropout(0.4)(x)
x=Dense(64,activation="relu",kernel_initializer='he_normal')(x)
x=Dropout(0.4)(x)
output = Dense(3, activation="softmax")(x)



model_vgg=Model(inp,output)

# **Model Summary**

In [ ]:
model_vgg.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg19 (Functional)              │ (None, 7, 7, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,098,369 (76.67 MB)

 Trainable params: 20,098,369 (76.67 MB)

 Non-trainable params: 0 (0.00 B)

# **Compile The Model Using Adam Optimizer**

In [ ]:
model_vgg.compile(optimizer=Adam(learning_rate=1e-6),loss='categorical_crossentropy',metrics=['accuracy'])


# **Perform Early Stopping to Reduce Overfitting**

In [ ]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5,verbose=1, restore_best_weights=True)

# **Train VGG Model**

In [ ]:
history_resnet=model_vgg.fit(train_ds,epochs=80,validation_data=val_ds,verbose=1,callbacks=[early_stopping])

Epoch 1/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.4092 - loss: 3.4259 - val_accuracy: 0.4880 - val_loss: 1.1350
Epoch 2/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 51s 994ms/step - accuracy: 0.4292 - loss: 1.7170 - val_accuracy: 0.5433 - val_loss: 0.9410
Epoch 3/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.4898 - loss: 1.2348 - val_accuracy: 0.6034 - val_loss: 0.8509
Epoch 4/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.4905 - loss: 1.1396 - val_accuracy: 0.6418 - val_loss: 0.7989
Epoch 5/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.4999 - loss: 1.0483 - val_accuracy: 0.6611 - val_loss: 0.7501
Epoch 6/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.5618 - loss: 0.9407 - val_accuracy: 0.7115 - val_loss: 0.6825
Epoch 7/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.6035 - loss: 0.8657 - val_accuracy: 0.7404 - val_loss: 0.6392
Epoch 8/80
51/51 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.6255 - loss: 0.8215 - val_accuracy: 0.7620 - val_lo

# **Save Our VGG Model in Local Environment**

In [ ]:
model_vgg.save("potato_vgg_full.h5")


converter = tf.lite.TFLiteConverter.from_keras_model(model_vgg)
tflite_model_vgg = converter.convert()

with open("potato_vgg_full.tflite", "wb") as f:
    f.write(tflite_model_vgg)


Saved artifact at '/tmp/tmp5cyc3bub'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_35')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136163430404688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911051472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911053776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911053008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911052432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911054352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911054928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911055696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911056656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136160911054544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13616091105

# **Load The VGG Model**

In [ ]:
model_resnet = load_model(
    "/content/drive/MyDrive/cnn_full_2model_resnet.h5"
)

# **Check The Accuracy And Loss**

In [ ]:
resnet_loss,resnet_acc=model_vgg.evaluate(test_ds)
print()
print()
print("Accuracy: ",resnet_acc)
print()
print("Loss: ",resnet_loss)

7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 645ms/step - accuracy: 0.9837 - loss: 0.0642


Accuracy:  0.9851852059364319

Loss:  0.05507618933916092


# **Evaluate Matrics**

In [ ]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

y_pred = np.argmax(model_vgg.predict(test_ds), axis=1)


print(" Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\n Classification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=['Early Blight', 'Healthy', 'Late Blight']
))


7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 330ms/step
 Confusion Matrix:
[[159   2   1]
 [  0 102   0]
 [  0   3 138]]

 Classification Report:
              precision    recall  f1-score   support

Early Blight       1.00      0.98      0.99       162
     Healthy       0.95      1.00      0.98       102
 Late Blight       0.99      0.98      0.99       141

    accuracy                           0.99       405
   macro avg       0.98      0.99      0.98       405
weighted avg       0.99      0.99      0.99       405

